In [7]:
import os
import sys
import sqlite3
from pathlib import Path
import importlib.util


# ==================================================
# 共通ユーティリティ
# ==================================================

def find_config_py(
    start: Path,
    max_up: int = 6,
) -> Path | None:
    """
    startから最大max_up階層まで上に遡り、
    utils/config.pyを探す。
    """
    cur = start.resolve()

    for _ in range(max_up + 1):

        candidate = (
            cur
            / "utils"
            / "config.py"
        )

        if candidate.is_file():
            return candidate

        cur = cur.parent

    return None


def import_config_from_path(
    cfg_path: Path,
):
    """
    ファイルパスから
    utils.configを読み込む。
    """

    spec = importlib.util.spec_from_file_location(
        "utils.config",
        cfg_path,
    )

    if spec and spec.loader:

        module = (
            importlib.util.module_from_spec(
                spec
            )
        )

        sys.modules[
            "utils.config"
        ] = module

        spec.loader.exec_module(
            module
        )

        return module

    raise ImportError(
        f"Failed to import config from: {cfg_path}"
    )


# ==================================================
# ベースディレクトリ決定
# ==================================================

try:

    base_dir = (
        Path(__file__)
        .resolve()
        .parent
    )

except NameError:

    base_dir = Path.cwd()


# ==================================================
# config 解決
# ==================================================

config_module = None

cfg_path = find_config_py(
    base_dir
)

if cfg_path:

    config_module = (
        import_config_from_path(
            cfg_path
        )
    )

else:

    sys.path.append(
        str(
            (
                base_dir
                / ".."
            ).resolve()
        )
    )

    try:

        from utils.config import PROJECT_DIR

    except Exception:

        PROJECT_DIR = None

    else:

        config_module = (
            sys.modules.get(
                "utils.config"
            )
        )


if config_module is None:

    PROJECT_DIR = os.getenv(
        "PROJECT_DIR",
        base_dir.name,
    )

else:

    PROJECT_DIR = getattr(
        config_module,
        "PROJECT_DIR",
        os.getenv(
            "PROJECT_DIR",
            base_dir.name,
        ),
    )


# ==================================================
# DBパス
# ==================================================

if os.name == "nt":

    home = Path(
        os.environ.get(
            "USERPROFILE",
            str(
                Path.home()
            )
        )
    )

else:

    home = Path.home()


user_base = (
    home
    / "myenv310"
    / PROJECT_DIR
)

db_dir = (
    user_base
    / "db"
)

db_dir.mkdir(
    parents=True,
    exist_ok=True,
)

db_path = (
    db_dir
    / "data.db"
)

columns_file = (
    db_dir
    / "columns_machine_master.txt"
)


# ==================================================
# columnsファイル探索
# ==================================================

if not columns_file.exists():

    project_root = (
        cfg_path.parent.parent
        if cfg_path
        else base_dir
    )

    alt_candidates = [

        project_root
        / "db"
        / "columns_machine_master.txt",

        base_dir
        / "db"
        / "columns_machine_master.txt",

        Path.cwd()
        / "db"
        / "columns_machine_master.txt",

    ]

    for candidate in alt_candidates:

        if candidate.exists():

            columns_file = candidate
            break


# ==================================================
# カラム定義読み込み
# ==================================================

if not columns_file.exists():

    raise FileNotFoundError(
        "columns_machine_master.txt が"
        f"見つかりません: {columns_file}"
    )


with columns_file.open(
    encoding="utf-8"
) as file:

    cols_types = [

        line.strip()

        for line in file

        if line.strip()
        and not line.lstrip().startswith(
            "#"
        )

    ]


if not cols_types:

    raise ValueError(
        "columns_machine_master.txt が空です"
    )


columns_definitions = ",\n    ".join(
    cols_types
)


# ==================================================
# テーブル名
# ==================================================

table_name = "machine_master"


# ==================================================
# CREATE TABLE
# ==================================================

create_table_sql = f"""

DROP TABLE IF EXISTS {table_name};

CREATE TABLE {table_name} (

    {columns_definitions}

);

"""


print("=" * 80)
print("[INFO] CREATE TABLE SQL")
print("=" * 80)

print(
    create_table_sql
)


# ==================================================
# DB作成
# ==================================================

connection = sqlite3.connect(
    str(
        db_path
    )
)

try:

    cursor = connection.cursor()

    cursor.executescript(
        create_table_sql
    )

    connection.commit()

finally:

    connection.close()


print("=" * 80)
print("[INFO] ✅ machine_master テーブル作成完了")
print(f"[INFO] DB : {db_path}")
print(f"[INFO] TABLE : {table_name}")
print(f"[INFO] PROJECT_DIR : {PROJECT_DIR}")
print(f"[INFO] COLUMN FILE : {columns_file}")
print("=" * 80)

[INFO] CREATE TABLE SQL


DROP TABLE IF EXISTS machine_master;

CREATE TABLE machine_master (

    id INTEGER PRIMARY KEY AUTOINCREMENT,
    master_machine_id TEXT NOT NULL UNIQUE,
    master_machine_category TEXT NOT NULL,
    master_machine_name TEXT NOT NULL,
    master_machine_maker TEXT,
    master_machine_model TEXT,
    master_machine_type TEXT,
    master_machine_gouki TEXT,
    master_machine_memo TEXT,
    master_machine_introduced_date TEXT,
    master_machine_game_system TEXT,
    master_machine_pworld_url TEXT,
    master_machine_pworld_image_url TEXT,
    master_machine_ptown_url TEXT,
    master_machine_ptown_image_url TEXT,
    master_machine_ptown_model TEXT,
    master_machine_ptown_name TEXT,
    master_machine_ptown_maker TEXT,
    master_machine_ptown_introduced_date TEXT,
    master_machine_pworld_normalized_model_search TEXT,
    master_machine_pworld_normalized_name_search TEXT,
    master_machine_ptown_normalized_model_search TEXT,
    master_machine_ptown_norm